In [1]:
# Cell 1: Install
!pip install -q lightgbm catboost scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python3.14 -m pip install --upgrade pip


In [2]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report

SEEDS = [42, 7, 123, 13, 99, 2024]
N_SPLITS = 10
print(f'Seeds: {SEEDS}')

Seeds: [42, 7, 123, 13, 99, 2024]


In [3]:
# Cell 3: Load data
TRAIN_DATA = pd.read_csv('train-data.csv', index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv', index_col='id')
print(f'Train: {TRAIN_DATA.shape}, Test: {TEST_DATA.shape}')

Train: (13249, 41), Test: (8834, 41)


In [4]:
# Cell 4: Preprocessing (identical to best run)
def preprocess(df):
    df = df.copy()
    drop_cols = [
        'first_name','last_name','insitute_name','institute_location',
        'test_1','test_2','test_3','test_4','test_5','treatment_consent',
        'birth_defects','alive'
    ]
    df = df.drop(columns=drop_cols)

    miss_cols = [
        'gender','maternal_defect','mother_age','father_age','respiration',
        'heart_rate','risk_level','place_birth','folic_acid','maternal_illness',
        'infertility_treatment','problem_previous_pregnancies','abortion_cnt',
        'white_blood_cell_count','blood_test',
        'symptom_1','symptom_2','symptom_3','symptom_4','symptom_5'
    ]
    df['missing_count']      = df[miss_cols].isna().sum(axis=1)
    df['missing_parent_age'] = df['mother_age'].isna().astype(int) + df['father_age'].isna().astype(int)
    df['missing_symptoms']   = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].isna().sum(axis=1)
    df['missing_clinical']   = df[['respiration','heart_rate','risk_level','blood_test']].isna().sum(axis=1)

    for col in ['mother_age','father_age','maternal_defect','gender','risk_level',
                'heart_rate','respiration','abortion_cnt','white_blood_cell_count']:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    binary_yn = [
        'mother_defect','father_defect','maternal_defect','paternal_defect',
        'folic_acid','maternal_illness','infertility_treatment',
        'problem_previous_pregnancies',
        'symptom_1','symptom_2','symptom_3','symptom_4','symptom_5'
    ]
    for col in binary_yn:
        df[col] = df[col].map({'Y':1,'N':0})

    df['respiration']  = df['respiration'].map({'A':1,'N':0})
    df['heart_rate']   = df['heart_rate'].map({'A':1,'N':0})
    df['risk_level']   = df['risk_level'].map({'H':1,'L':0})
    df['place_birth']  = df['place_birth'].map({'I':1,'H':0})
    df['gender']       = df['gender'].map({'M':0,'F':1,'A':2})
    df['autopsy']      = df['autopsy'].map({'Y':1,'N':0})
    for col in ['birth_asphyxia','radiation_exposure','substance_abuse']:
        df[col] = df[col].map({'Y':1,'N':0,'NR':2})
    df['blood_test'] = df['blood_test'].map({'N':0,'I':1,'S':2,'A':3})

    df['defect_sum']           = df[['mother_defect','father_defect','maternal_defect','paternal_defect']].sum(axis=1)
    df['symptom_sum']          = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].sum(axis=1)
    df['defect_x_symptom']     = df['defect_sum'] * df['symptom_sum']
    df['any_defect']           = (df['defect_sum'] > 0).astype(int)
    df['any_symptom']          = (df['symptom_sum'] > 0).astype(int)
    df['high_symptom']         = (df['symptom_sum'] >= 4).astype(int)
    df['all_defects']          = (df['defect_sum'] == 4).astype(int)
    df['parent_age_gap']       = (df['father_age'] - df['mother_age']).abs()
    df['symptom_defect_ratio'] = df['symptom_sum'] / (df['defect_sum'] + 1)
    df['s4_and_s5']            = ((df['symptom_4']==1) & (df['symptom_5']==1)).astype(int)
    df['no_s4_s5']             = ((df['symptom_4']==0) & (df['symptom_5']==0)).astype(int)
    df['late_vs_early']        = (df['symptom_4'].fillna(0) + df['symptom_5'].fillna(0)
                                  - df['symptom_1'].fillna(0) - df['symptom_2'].fillna(0))
    df['weighted_sym']         = (df['symptom_1'].fillna(0)*1 + df['symptom_2'].fillna(0)*1 +
                                  df['symptom_3'].fillna(0)*1 + df['symptom_4'].fillna(0)*2 +
                                  df['symptom_5'].fillna(0)*2)
    df['both_parents_defect']  = ((df['mother_defect']==1) & (df['father_defect']==1)).astype(int)
    df['no_parent_defect']     = ((df['mother_defect']==0) & (df['father_defect']==0)).astype(int)
    df['zero_symptom']         = (df['symptom_sum'] == 0).astype(int)
    df['zero_defect']          = (df['defect_sum']  == 0).astype(int)
    df['zero_both']            = ((df['symptom_sum']==0) & (df['defect_sum']==0)).astype(int)
    df['very_low_sym']         = (df['symptom_sum'] <= 1).astype(int)

    df = df.fillna(-1)
    return df

X_train_raw = preprocess(TRAIN_DATA)
X_test_raw  = preprocess(TEST_DATA)
y_train     = TRAIN_LABEL['disorder'].values

DROP_ADV = ['blood_cell_count','white_blood_cell_count','mother_age']
X_train = X_train_raw.drop(columns=DROP_ADV)
X_test  = X_test_raw.drop(columns=DROP_ADV)
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

X_train: (13249, 58), X_test: (8834, 58)


In [5]:
# Cell 5: Params
class_counts  = np.bincount(y_train)
class_weights = len(y_train) / (10 * class_counts)

CAT_PARAMS = {
    'iterations'           : 3000,
    'learning_rate'        : 0.03,
    'depth'                : 8,
    'l2_leaf_reg'          : 1.2554074561515052,
    'random_strength'      : 0.15300699009014024,
    'rsm'                  : 0.6693037260566354,
    'bagging_temperature'  : 0.7561917100518147,
    'min_data_in_leaf'     : 23,
    'early_stopping_rounds': 150,
    'eval_metric'          : 'TotalF1:average=Macro',
    'verbose'              : 0,
    'thread_count'         : -1,
}

# LGB params — heavy regularisation to match our low-OOF philosophy
LGB_PARAMS = {
    'objective'       : 'multiclass',
    'num_class'       : 10,
    'metric'          : 'multi_logloss',
    'n_estimators'    : 3000,
    'learning_rate'   : 0.03,
    'num_leaves'      : 31,        # conservative, avoids overfitting
    'max_depth'       : 6,
    'min_child_samples': 30,       # similar to min_data_in_leaf
    'subsample'       : 0.8,
    'subsample_freq'  : 1,
    'colsample_bytree': 0.7,
    'reg_alpha'       : 0.1,
    'reg_lambda'      : 1.0,
    'n_jobs'          : -1,
    'verbose'         : -1,
}

print('Params ready.')

Params ready.


In [6]:
# Cell 6: Train both models
def run_model(model_name, seeds, X_train, y_train, X_test, class_weights):
    all_oof  = np.zeros((len(y_train), 10))
    all_test = np.zeros((len(X_test), 10))
    seed_scores = []

    for SEED in seeds:
        print(f"\n{'='*35} {model_name} SEED={SEED} {'='*35}")
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
        oof  = np.zeros((len(y_train), 10))
        test = np.zeros((len(X_test), 10))
        fold_scores = []

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]
            sw = np.array([class_weights[c] for c in y_tr])

            if model_name == 'CatBoost':
                model = CatBoostClassifier(class_weights=class_weights, random_seed=SEED, **CAT_PARAMS)
                model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
                val_proba  = model.predict_proba(X_val)
                test_proba = model.predict_proba(X_test)

            else:  # LightGBM
                sw_val = np.array([class_weights[c] for c in y_val])
                model = lgb.LGBMClassifier(random_state=SEED, class_weight='balanced', **LGB_PARAMS)
                model.fit(X_tr, y_tr,
                          eval_set=[(X_val, y_val)],
                          callbacks=[lgb.early_stopping(150, verbose=False),
                                     lgb.log_evaluation(-1)])
                val_proba  = model.predict_proba(X_val)
                test_proba = model.predict_proba(X_test)

            score = balanced_accuracy_score(y_val, np.argmax(val_proba, axis=1))
            fold_scores.append(score)
            print(f'  Fold {fold+1:2d}: BA={score:.4f}  best_iter={model.best_iteration_}')

            oof[val_idx] += val_proba
            test += test_proba / N_SPLITS

        oof_score = balanced_accuracy_score(y_train, np.argmax(oof, axis=1))
        seed_scores.append(oof_score)
        print(f'  OOF (seed={SEED}): {oof_score:.4f} | mean={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')

        all_oof  += oof / len(seeds)
        all_test += test / len(seeds)

    final_oof = balanced_accuracy_score(y_train, np.argmax(all_oof, axis=1))
    print(f"\n{model_name} FINAL OOF: {final_oof:.4f} | seed std: {np.std(seed_scores):.4f}")
    return all_oof, all_test, final_oof

cat_oof, cat_test, cat_final = run_model('CatBoost', SEEDS, X_train, y_train, X_test, class_weights)
lgb_oof, lgb_test, lgb_final = run_model('LightGBM', SEEDS, X_train, y_train, X_test, class_weights)


=================================== CatBoost SEED=42 ===================================
  Fold  1: BA=0.3755  best_iter=44
  Fold  2: BA=0.4059  best_iter=31
  Fold  3: BA=0.4214  best_iter=81
  Fold  4: BA=0.4057  best_iter=15
  Fold  5: BA=0.4040  best_iter=86
  Fold  6: BA=0.4443  best_iter=61
  Fold  7: BA=0.3918  best_iter=8
  Fold  8: BA=0.3835  best_iter=366
  Fold  9: BA=0.3995  best_iter=76
  Fold 10: BA=0.4514  best_iter=74
  OOF (seed=42): 0.4089 | mean=0.4083 ± 0.0232

=================================== CatBoost SEED=7 ===================================
  Fold  1: BA=0.4129  best_iter=53
  Fold  2: BA=0.4010  best_iter=73
  Fold  3: BA=0.4101  best_iter=99
  Fold  4: BA=0.4042  best_iter=223
  Fold  5: BA=0.3769  best_iter=9
  Fold  6: BA=0.4186  best_iter=53
  Fold  7: BA=0.3841  best_iter=56
  Fold  8: BA=0.3608  best_iter=2
  Fold  9: BA=0.4045  best_iter=270
  Fold 10: BA=0.3816  best_iter=50
  OOF (seed=7): 0.3959 | mean=0.3955 ± 0.0176

===========================

In [7]:
# Cell 7: Blend 50/50 and evaluate
blend_oof  = 0.5 * cat_oof  + 0.5 * lgb_oof
blend_test = 0.5 * cat_test + 0.5 * lgb_test

blend_oof_score = balanced_accuracy_score(y_train, np.argmax(blend_oof, axis=1))

print(f"{'='*60}")
print(f'CatBoost OOF:  {cat_final:.4f}')
print(f'LightGBM OOF:  {lgb_final:.4f}')
print(f'Blend OOF:     {blend_oof_score:.4f}')
print(f'Current best LB: 0.37551')
print(f'Predicted LB (OOF - 0.013): {blend_oof_score - 0.013:.4f}')
print(f"{'='*60}")

# Per-class recall
best_recall = {0:0.414, 1:0.374, 2:0.317, 3:0.332, 4:0.672,
               5:0.312, 6:0.523, 7:0.248, 8:0.538, 9:0.158}
report = classification_report(y_train, np.argmax(blend_oof, axis=1), output_dict=True)
print(f'\n{"Class":<6} {"best run":>10} {"blend":>8} {"Δ":>7}')
print('-' * 36)
for cls in range(10):
    r     = report[str(cls)]['recall']
    r_old = best_recall[cls]
    delta = r - r_old
    flag  = ' ← UP' if delta > 0.02 else (' ← LOW' if r < 0.25 else '')
    print(f'{cls:<6} {r_old:>10.3f} {r:>8.3f} {delta:>+7.3f}{flag}')

CatBoost OOF:  0.3889
LightGBM OOF:  0.3316
Blend OOF:     0.3521
Current best LB: 0.37551
Predicted LB (OOF - 0.013): 0.3391

Class    best run    blend       Δ
------------------------------------
0           0.414    0.388  -0.026
1           0.374    0.399  +0.025 ← UP
2           0.317    0.370  +0.053 ← UP
3           0.332    0.359  +0.027 ← UP
4           0.672    0.431  -0.241
5           0.312    0.348  +0.036 ← UP
6           0.523    0.550  +0.027 ← UP
7           0.248    0.243  -0.005 ← LOW
8           0.538    0.319  -0.219
9           0.158    0.115  -0.043 ← LOW


In [8]:
# Cell 8: Save
final_preds = np.argmax(blend_test, axis=1)
submission  = pd.DataFrame({'id': TEST_DATA.index, 'disorder': final_preds}).set_index('id')
submission.to_csv('submission_blend.csv')
print('Saved: submission_blend.csv')
print(f'\nBlend OOF:  {blend_oof_score:.4f}')
print(f'Current best LB: 0.37551')
print(f'\nSubmit ONLY if blend OOF < 0.3949 (lower = better here!)')

Saved: submission_blend.csv

Blend OOF:  0.3521
Current best LB: 0.37551

Submit ONLY if blend OOF < 0.3949 (lower = better here!)
